In [ ]:
!gdown --id 1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5 -O /content/stocks_df_combined_2026_09_18.parquet.brotli

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5
From (redirected): https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5&confirm=t&uuid=44eee2ea-7ef5-480a-b1d2-5603b7414a68
To: /content/stocks_df_combined_2026_09_18.parquet.brotli
100% 144M/144M [00:01<00:00, 97.6MB/s]


In [ ]:
"""
Module 4 Homework - All Questions Combined
==========================================
Q1: Less Features is More (CAGR for pred10_rf_best_rule_55 with fewer dummies)
Q2: Most Efficient RF Decision Rule (threshold with avg trades/day < 5)
Q3: Redefining Growth Target (number of trades for pred5_clf_10 with new target)
Q4: Exploratory - describe ideal trading strategy

Prerequisites:
  pip install pandas numpy scikit-learn pyarrow

Data: Download from Google Drive:
  gdown https://drive.google.com/file/d/1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5/view?usp=sharing -O stocks_df_combined_2026_09_18.parquet.brotli

Usage:
  python3 module04_homework_all.py --data-dir /path/to/data
"""
import numpy as np
import pandas as pd
import warnings
import gc
import argparse
import os

warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# ============================================================
# COMMON FUNCTIONS & SETUP
# ============================================================

TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1','aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo','dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']

MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
 'DGS1', 'DGS5', 'DGS10']

CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative',
                    'volatility', 'ln_volume','div_payout','stock_split']


def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    split_labels = []
    for date in df['Date']:
        if date <= train_end:
            split_labels.append('train')
        elif date <= val_end:
            split_labels.append('validation')
        else:
            split_labels.append('test')
    df['split'] = split_labels
    return df


def clean_dataframe_from_inf_and_nan(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    return df


def fit_decision_tree(X, y, max_depth=20):
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf.fit(X, y)
    return clf, X.columns


def tpr_fpr_dataframe(y_true, y_pred, only_even=False):
    scores = []
    if only_even == False:
        thresholds = np.linspace(0, 1, 101)
    else:
        thresholds = np.linspace(0, 1, 51)
    for t in thresholds:
        actual_positive = (y_true == 1)
        actual_negative = (y_true == 0)
        predict_positive = (y_pred >= t)
        predict_negative = (y_pred < t)
        tp = (predict_positive & actual_positive).sum()
        tn = (predict_negative & actual_negative).sum()
        fp = (predict_positive & actual_negative).sum()
        fn = (predict_negative & actual_positive).sum()
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1_score = 2*precision*recall / (precision+recall) if precision+recall > 0 else 0
        accuracy = (tp+tn) / (tp+tn+fp+fn)
        scores.append((t, tp, fp, fn, tn, precision, recall, accuracy, f1_score))
    columns = ['threshold', 'tp', 'fp', 'fn', 'tn','precision','recall', 'accuracy','f1_score']
    df_scores = pd.DataFrame(scores, columns=columns)
    df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
    df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)
    return df_scores


def load_and_prepare_data(data_path, use_ticker=True, use_month_wom=True,
                          redefine_target_threshold=None):
    """Load data, define variables, generate dummies, temporal split.

    Args:
        data_path: Path to the parquet file
        use_ticker: Whether to include 'Ticker' in CATEGORICAL
        use_month_wom: Whether to create and include 'month_wom' in CATEGORICAL
        redefine_target_threshold: If set, redefine is_positive_growth_30d_future
                                   to be 1 when growth_future_30d > this threshold

    Returns:
        new_df, NUMERICAL, DUMMIES, features_list
    """
    print("Loading data...")
    df_full = pd.read_parquet(data_path)
    print(f"Loaded {len(df_full)} records, {len(df_full.columns)} columns")

    # Define variable sets
    GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]

    # CATEGORICAL setup
    CATEGORICAL = ['Month', 'Weekday', 'ticker_type']
    if use_ticker:
        CATEGORICAL.insert(2, 'Ticker')  # Insert Ticker before ticker_type

    TO_PREDICT = [g for g in df_full.keys() if (g.find('future')>=0)]

    # Custom features
    df_full['ln_volume'] = df_full.Volume.apply(lambda x: np.log(x+ 1e-6))
    df_full['div_payout'] = (df_full.Dividends>0).astype(int)
    df_full['stock_split'] = (df_full['Stock Splits']>0).astype(int)

    TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]
    NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

    # Q3: Optionally redefine target
    if redefine_target_threshold is not None:
        print(f"\nRedefining is_positive_growth_30d_future: threshold = {redefine_target_threshold}")
        print(f"  Original class 1 %: {df_full['is_positive_growth_30d_future'].mean():.4f}")
        df_full['is_positive_growth_30d_future'] = (df_full['growth_future_30d'] > redefine_target_threshold).astype(int)
        print(f"  New class 1 %: {df_full['is_positive_growth_30d_future'].mean():.4f}")

    # Truncate to 25 years
    df = df_full[df_full.Date>='2000-01-01'].copy()
    del df_full
    gc.collect()
    print(f"Truncated to {len(df)} records (25 years)")

    # Convert categorical columns
    df.loc[:,'Month'] = pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
    df.loc[:,'Weekday'] = df['Weekday'].astype('string')

    # Optionally create month_wom
    if use_month_wom:
        df.loc[:,'wom'] = df.Date.apply(lambda d: (d.day-1)//7 + 1)
        df.loc[:,'wom'] = df.loc[:,'wom'].astype(str)
        df.loc[:,'month_wom'] = df.Month + '_w' + df.wom
        del df['wom']
        CATEGORICAL.append('month_wom')

    print(f"CATEGORICAL = {CATEGORICAL}")

    # Generate dummies
    dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
    DUMMIES = dummy_variables.keys().to_list()
    df_with_dummies = pd.concat([df, dummy_variables], axis=1)
    del df, dummy_variables
    gc.collect()

    num_features = len(NUMERICAL) + len(DUMMIES)
    print(f"NUMERICAL: {len(NUMERICAL)}, DUMMIES: {len(DUMMIES)}, TOTAL: {num_features}")

    # Temporal split
    min_date_df = df_with_dummies.Date.min()
    max_date_df = df_with_dummies.Date.max()
    df_with_dummies = temporal_split(df_with_dummies, min_date=min_date_df, max_date=max_date_df)
    new_df = df_with_dummies.copy()
    del df_with_dummies
    gc.collect()

    features_list = NUMERICAL + DUMMIES
    return new_df, NUMERICAL, DUMMIES, features_list


def add_manual_predictions(new_df):
    """Add manual prediction columns (pred0-pred4)."""
    new_df['pred0_manual_cci'] = (new_df.cci>200).astype(int)
    new_df['pred1_manual_prev_g1'] = (new_df.growth_30d>1).astype(int)
    new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)
    new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <= 4.5) & (new_df['DGS5'] <= 4)).astype(int)
    new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS'] <= 4.795)).astype(int)
    return new_df


def train_models_and_predict(new_df, features_list, add_rf_thresholds=None):
    """Train Decision Tree and Random Forest models, add predictions.

    Args:
        add_rf_thresholds: List of threshold ints (e.g., [56,57,58,59,60,61,62,63,64])
                           to add extra RF threshold rules. None = no extra rules.

    Returns:
        new_df with prediction columns added
    """
    to_predict = 'is_positive_growth_30d_future'

    train_valid_df = new_df[new_df.split.isin(['train','validation'])]
    test_df = new_df[new_df.split.isin(['test'])]

    X_train_valid = train_valid_df[features_list+[to_predict]].copy(deep=True)
    X_test = test_df[features_list+[to_predict]].copy(deep=True)
    X_all = new_df[features_list+[to_predict]].copy(deep=True)
    del train_valid_df, test_df
    gc.collect()

    X_train_valid = clean_dataframe_from_inf_and_nan(X_train_valid)
    X_test = clean_dataframe_from_inf_and_nan(X_test)
    X_all = clean_dataframe_from_inf_and_nan(X_all)

    y_train_valid = X_train_valid[to_predict]
    y_test = X_test[to_predict]

    X_train_valid_features = X_train_valid[features_list]
    X_test_features = X_test[features_list]
    X_all_features = X_all[features_list]

    # Decision Tree (max_depth=10) -> pred5_clf_10
    print("\nTraining Decision Tree (max_depth=10)...")
    clf_10, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=10)
    new_df['pred5_clf_10'] = clf_10.predict(X_all_features)

    # Decision Tree (max_depth=4) -> pred6_clf_4
    print("Training Decision Tree (max_depth=4)...")
    clf_4, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=4)
    new_df['pred6_clf_4'] = clf_4.predict(X_all_features)

    # Decision Tree (max_depth=7) -> pred7_clf_second_best
    print("Training Decision Tree (max_depth=7)...")
    clf_best, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=7)
    new_df['pred7_clf_second_best'] = clf_best.predict(X_all_features)

    # Random Forest
    print("\nTraining Random Forest (n_estimators=100, max_depth=13)...")
    rf_best = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1)
    rf_best = rf_best.fit(X_train_valid_features, y_train_valid)
    print("Random Forest trained.")

    # Decision Tree threshold rules
    y_pred_all_clf = clf_best.predict_proba(X_all_features)
    y_pred_all_clf_class1 = np.array([k[1] for k in y_pred_all_clf])
    new_df['proba_pred8'] = y_pred_all_clf_class1
    new_df['pred8_clf_second_best_rule_56'] = (y_pred_all_clf_class1 >= 0.56).astype(int)
    new_df['proba_pred9'] = y_pred_all_clf_class1
    new_df['pred9_clf_second_best_rule_66'] = (y_pred_all_clf_class1 >= 0.66).astype(int)

    # Random Forest threshold rules (original: 0.55 and 0.65)
    y_pred_all_rf = rf_best.predict_proba(X_all_features)
    y_pred_all_rf_class1 = np.array([k[1] for k in y_pred_all_rf])
    new_df['proba_pred10'] = y_pred_all_rf_class1
    new_df['pred10_rf_best_rule_55'] = (y_pred_all_rf_class1 >= 0.55).astype(int)
    new_df['proba_pred11'] = y_pred_all_rf_class1
    new_df['pred11_rf_best_rule_65'] = (y_pred_all_rf_class1 >= 0.65).astype(int)

    # Optional: add extra RF threshold rules (for Q2)
    rf_tpr_df = None
    if add_rf_thresholds is not None:
        # Build tpr_fpr_dataframe for Random Forest (Q2 requirement)
        y_pred_test_rf = rf_best.predict_proba(X_test_features)
        y_pred_test_rf_class1 = np.array([k[1] for k in y_pred_test_rf])
        rf_tpr_df = tpr_fpr_dataframe(y_test, y_pred_test_rf_class1, only_even=True)

        pred_num = 12
        for threshold_int in add_rf_thresholds:
            threshold = threshold_int / 100.0
            col_name = f'pred{pred_num}_rf_best_rule_{threshold_int}'
            new_df[col_name] = (y_pred_all_rf_class1 >= threshold).astype(int)
            print(f"  Added {col_name} with threshold {threshold}")
            pred_num += 1

    # Clean up
    del clf_10, clf_4, clf_best, rf_best
    del X_train_valid, X_test, X_all
    del X_train_valid_features, X_test_features, X_all_features
    del y_pred_all_clf, y_pred_all_clf_class1, y_pred_all_rf, y_pred_all_rf_class1
    gc.collect()

    return new_df, rf_tpr_df


def run_trading_simulations(new_df):
    """Run trading simulations for all predictions."""
    PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
    print(f"\nPredictions: {PREDICTIONS}")
    print("\n=== Running Trading Simulations ===")

    sim1_results = []
    for pred in PREDICTIONS:
        pred_prefix = pred.split('_')[0]
        new_df['sim1_gross_rev_'+pred_prefix] = new_df[pred] * 100 * (new_df['growth_future_30d']-1)
        new_df['sim1_fees_'+pred_prefix] = -new_df[pred] * 100 * 0.002
        new_df['sim1_net_rev_'+pred_prefix] = new_df['sim1_gross_rev_'+pred_prefix] + new_df['sim1_fees_'+pred_prefix]

        filter_test_and_positive_pred = (new_df.split=='test')&(new_df[pred]==1)
        sim1_count_investments = len(new_df[filter_test_and_positive_pred])
        sim1_gross_rev = new_df[filter_test_and_positive_pred]['sim1_gross_rev_'+pred_prefix].sum()
        sim1_fees = new_df[filter_test_and_positive_pred]['sim1_fees_'+pred_prefix].sum()
        sim1_net_rev = new_df[filter_test_and_positive_pred]['sim1_net_rev_'+pred_prefix].sum()

        sim1_fees_percentage = -sim1_fees/sim1_gross_rev if sim1_gross_rev>0 else None
        sim1_average_net_revenue = sim1_net_rev/sim1_count_investments if sim1_count_investments>0 else None

        df_investments_count_daily = pd.DataFrame(new_df[filter_test_and_positive_pred].groupby('Date')[pred].count())
        sim1_avg_investments_per_day = df_investments_count_daily[pred].mean()
        sim1_q75_investments_per_day = df_investments_count_daily[pred].quantile(0.75)
        sim1_capital = 100 * 30 * sim1_q75_investments_per_day
        sim1_CAGR = ((sim1_capital+sim1_net_rev)/sim1_capital)**(1/4)

        sim1_results.append((pred, sim1_count_investments, sim1_gross_rev, sim1_fees, sim1_net_rev,
                             sim1_fees_percentage, sim1_average_net_revenue, sim1_avg_investments_per_day,
                             sim1_capital, sim1_CAGR))

    columns_simulation = ['prediction', 'sim1_count_investments', 'sim1_gross_rev', 'sim1_fees', 'sim1_net_rev',
                           'sim1_fees_percentage','sim1_average_net_revenue','sim1_avg_investments_per_day',
                           'sim1_capital','sim1_CAGR']

    df_sim1_results = pd.DataFrame(sim1_results, columns=columns_simulation)
    df_sim1_results['sim1_growth_capital_4y'] = (df_sim1_results.sim1_net_rev+df_sim1_results.sim1_capital) / df_sim1_results.sim1_capital
    return df_sim1_results


# ============================================================
# QUESTION 1: Less Features is More
# ============================================================
def answer_q1(data_path):
    """Q1: Remove Ticker and month_wom from CATEGORICAL -> find CAGR for pred10_rf_best_rule_55."""
    print("\n" + "="*70)
    print("QUESTION 1: Less Features is More")
    print("="*70)

    # Load with fewer features: no Ticker, no month_wom
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(
        data_path, use_ticker=False, use_month_wom=False
    )

    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list)
    df_sim1_results = run_trading_simulations(new_df)

    # Extract answer
    pred10_result = df_sim1_results[df_sim1_results.prediction == 'pred10_rf_best_rule_55']

    print("\n" + "-"*50)
    print("Q1 RESULTS:")
    print(df_sim1_results[['prediction','sim1_count_investments','sim1_CAGR','sim1_avg_investments_per_day']].to_string())

    if len(pred10_result) > 0:
        cagr = pred10_result['sim1_CAGR'].values[0]
        avg_invest = pred10_result['sim1_avg_investments_per_day'].values[0]
        count = pred10_result['sim1_count_investments'].values[0]
        print(f"\n*** Q1 ANSWER: CAGR for pred10_rf_best_rule_55 = {cagr:.4f} ***")
        print(f"    Avg investments/day: {avg_invest:.2f}, Total: {int(count)}")
    else:
        print("ERROR: pred10_rf_best_rule_55 not found!")

    del new_df
    gc.collect()
    return cagr if len(pred10_result) > 0 else None


# ============================================================
# QUESTION 2: Most Efficient Random Forest Decision Rule
# ============================================================
def answer_q2(data_path):
    """Q2: Add RF threshold rules 0.56-0.64, find second-best with avg trades/day < 5."""
    print("\n" + "="*70)
    print("QUESTION 2: Most Efficient Random Forest Decision Rule")
    print("="*70)

    # Same setup as Q1: fewer features (no Ticker, no month_wom)
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(
        data_path, use_ticker=False, use_month_wom=False
    )

    new_df = add_manual_predictions(new_df)
    # Add extra RF threshold rules: 0.56, 0.57, ..., 0.64
    new_df, rf_tpr_df = train_models_and_predict(
        new_df, features_list, add_rf_thresholds=list(range(56, 65))
    )

    # Show tpr_fpr_dataframe for Random Forest
    if rf_tpr_df is not None:
        print("\ntpr_fpr_dataframe for Random Forest (thresholds 0.50-0.70):")
        print(rf_tpr_df[(rf_tpr_df.threshold>=0.50) & (rf_tpr_df.threshold<=0.70)]
              [['threshold','tp','fp','precision','recall']].to_string())

    df_sim1_results = run_trading_simulations(new_df)

    # Filter RF rules with avg trades/day < 5
    rf_results = df_sim1_results[df_sim1_results.prediction.str.contains('rf_best_rule')]
    rf_under5 = rf_results[rf_results.sim1_avg_investments_per_day < 5].sort_values('sim1_CAGR', ascending=False)

    print("\n" + "-"*50)
    print("Q2 RESULTS: ALL RF rules")
    print(rf_results[['prediction','sim1_count_investments','sim1_CAGR','sim1_avg_investments_per_day']].to_string())
    print("\nQ2: RF rules with avg trades/day < 5 (sorted by CAGR desc):")
    print(rf_under5[['prediction','sim1_count_investments','sim1_CAGR','sim1_avg_investments_per_day']].to_string())

    if len(rf_under5) > 0:
        best = rf_under5.iloc[0]
        threshold_str = best['prediction'].split('rule_')[-1]
        threshold = int(threshold_str) / 100.0
        print(f"\n*** Q2 ANSWER: Threshold = {threshold:.2f} (pred rule {threshold_str}) ***")
        print(f"    CAGR: {best['sim1_CAGR']:.4f}, Avg trades/day: {best['sim1_avg_investments_per_day']:.2f}")
    else:
        print("ERROR: No RF rule found with avg trades/day < 5!")
        threshold = None

    del new_df
    gc.collect()
    return threshold


# ============================================================
# QUESTION 3: Redefining the Growth Target
# ============================================================
def answer_q3(data_path):
    """Q3: Redefine is_positive_growth_30d_future with threshold 1.022, find trades for pred5_clf_10."""
    print("\n" + "="*70)
    print("QUESTION 3: Redefining the Growth Target")
    print("="*70)

    # ORIGINAL notebook setup (with Ticker and month_wom), but redefined target
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(
        data_path, use_ticker=True, use_month_wom=True,
        redefine_target_threshold=1.022
    )

    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list)
    df_sim1_results = run_trading_simulations(new_df)

    # Find best predictor and pred5_clf_10 specifically
    best_pred = df_sim1_results.loc[df_sim1_results['sim1_CAGR'].idxmax()]
    pred5_result = df_sim1_results[df_sim1_results.prediction == 'pred5_clf_10']

    print("\n" + "-"*50)
    print("Q3 RESULTS:")
    print(df_sim1_results[['prediction','sim1_count_investments','sim1_CAGR','sim1_avg_investments_per_day']].to_string())
    print(f"\nBest predictor by CAGR: {best_pred['prediction']} (CAGR={best_pred['sim1_CAGR']:.4f})")

    if len(pred5_result) > 0:
        count = int(pred5_result['sim1_count_investments'].values[0])
        cagr = pred5_result['sim1_CAGR'].values[0]
        print(f"\n*** Q3 ANSWER: Number of trades for pred5_clf_10 = {count} ***")
        print(f"    CAGR: {cagr:.4f}, Avg investments/day: {pred5_result['sim1_avg_investments_per_day'].values[0]:.2f}")
    else:
        print("ERROR: pred5_clf_10 not found!")
        count = None

    del new_df
    gc.collect()
    return count


# ============================================================
# QUESTION 4: Ideal Trading Strategy (Exploratory)
# ============================================================
def answer_q4():
    """Q4: Describe ideal trading strategy."""
    print("\n" + "="*70)
    print("QUESTION 4: Ideal Trading Strategy (Exploratory)")
    print("="*70)

    strategy = """
My ideal strategy would combine multiple models and manage position sizing based on prediction confidence:

1. CONFIDENCE-WEIGHTED ALLOCATION: Instead of investing a flat $100 per positive prediction,
   I would use the prediction probability from the Random Forest to size positions. For example,
   invest $50 + $100 x (probability - 0.556) for predictions above the 0.56 threshold, so
   higher-confidence predictions get more capital.

2. MULTI-MODEL CONSENSUS: Only invest when at least 2 out of 3 models agree (e.g.,
   Random Forest at threshold 0.57, Decision Tree at depth 10, and the manual DGS10 rule).
   This reduces false positives and trading frequency while maintaining precision.

3. SECTOR DIVERSIFICATION CAP: Limit maximum exposure to 3 stocks per day to reduce
   concentration risk, selecting the ones with the highest prediction probabilities.

4. DYNAMIC THRESHOLD ADJUSTMENT: Use a higher threshold (e.g., 0.60) during high-volatility
   periods (when VIX or ATR indicators are elevated) and a lower threshold (0.55) during
   calm markets, adapting risk to market conditions.

This approach aims to improve risk-adjusted returns by being more selective and sizing
positions based on conviction, rather than treating all positive predictions equally.
"""
    print(strategy)
    return strategy


# ============================================================
# MAIN / NOTEBOOK EXECUTION ENTRYPOINT
# ============================================================
if __name__ == '__main__':
    # Default to downloaded file paths in Colab directory structure
    data_path = '/content/stocks_df_combined_2026_09_18.parquet.brotli'

    if not os.path.exists(data_path):
        # Fallback to alternate local directory
        data_path = 'stocks_df_combined_2026_09_18.parquet.brotli'

    if not os.path.exists(data_path):
        print(f"ERROR: Data file not found!")
        print("Please run: !gdown https://drive.google.com/file/d/1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5/view?usp=sharing -O stocks_df_combined_2026_09_18.parquet.brotli")
    else:
        answers = {}

        # Execute each question
        answers['Q1'] = answer_q1(data_path)
        answers['Q2'] = answer_q2(data_path)
        answers['Q3'] = answer_q3(data_path)
        answers['Q4'] = answer_q4()

        # Final Summary
        print("\n" + "="*70)
        print("FINAL ANSWERS SUMMARY")
        print("="*70)
        if 'Q1' in answers and answers['Q1'] is not None:
            print(f"  Q1 (CAGR for pred10_rf_best_rule_55 with fewer features): {answers['Q1']:.4f}")
        if 'Q2' in answers and answers['Q2'] is not None:
            print(f"  Q2 (RF threshold with avg trades/day < 5):               {answers['Q2']:.2f}")
        if 'Q3' in answers and answers['Q3'] is not None:
            print(f"  Q3 (Number of trades for pred5_clf_10 with new target):   {answers['Q3']}")
        if 'Q4' in answers and answers['Q4'] is not None:
            print(f"  Q4 (Ideal trading strategy):                              See description above")


QUESTION 1: Less Features is More
Loading data...


ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
# Re-running the homework script using functions from the main cell.
# Make sure you have run the cell above (Ha9895-X-a2n) first to define the functions!
data_path = '/content/stocks_df_combined_2026_09_18.parquet.brotli'

try:
    answers = {}
    answers['Q1'] = answer_q1(data_path)
    answers['Q2'] = answer_q2(data_path)
    answers['Q3'] = answer_q3(data_path)
    answers['Q4'] = answer_q4()

    print("\n" + "="*70)
    print("FINAL ANSWERS SUMMARY")
    print("="*70)
    if 'Q1' in answers and answers['Q1'] is not None:
        print(f"  Q1 (CAGR for pred10_rf_best_rule_55 with fewer features): {answers['Q1']:.4f}")
    if 'Q2' in answers and answers['Q2'] is not None:
        print(f"  Q2 (RF threshold with avg trades/day < 5):               {answers['Q2']:.2f}")
    if 'Q3' in answers and answers['Q3'] is not None:
        print(f"  Q3 (Number of trades for pred5_clf_10 with new target):   {answers['Q3']}")
except NameError:
    print("NameError: The functions are not defined. Please run the cell above (Ha9895-X-a2n) first, and then run this cell.")

NameError: The functions are not defined. Please run the cell above (Ha9895-X-a2n) first, and then run this cell.


In [ ]:
data_path = '/content/stocks_df_combined_2026_09_18.parquet.brotli'

answers = {}
answers['Q1'] = answer_q1(data_path)
answers['Q2'] = answer_q2(data_path)
answers['Q3'] = answer_q3(data_path)
answers['Q4'] = answer_q4()

print("\n" + "="*70)
print("FINAL ANSWERS SUMMARY")
print("="*70)
if 'Q1' in answers and answers['Q1'] is not None:
    print(f"  Q1 (CAGR for pred10_rf_best_rule_55 with fewer features): {answers['Q1']:.4f}")
if 'Q2' in answers and answers['Q2'] is not None:
    print(f"  Q2 (RF threshold with avg trades/day < 5):               {answers['Q2']:.2f}")
if 'Q3' in answers and answers['Q3'] is not None:
    print(f"  Q3 (Number of trades for pred5_clf_10 with new target):   {answers['Q3']}")


FINAL ANSWERS SUMMARY
  Q1 (CAGR for pred10_rf_best_rule_55 with fewer features): 1.1359
  Q2 (RF threshold with avg trades/day < 5):               0.56
  Q3 (Number of trades for pred5_clf_10 with new target):   7337


In [ ]:
import os

file_path = 'stocks_df_combined_2026_09_18.parquet.brotli'
absolute_path = '/content/stocks_df_combined_2026_09_18.parquet.brotli'

print(f"File exists in current directory: {os.path.exists(file_path)}")
print(f"File exists at absolute path: {os.path.exists(absolute_path)}")

if os.path.exists(absolute_path):
    size_mb = os.path.getsize(absolute_path) / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")
else:
    print("Warning: File not found.")

File exists in current directory: True
File exists at absolute path: True
File size: 137.18 MB


In [ ]:
# Programmatically read and define all functions from the main cell block 'Ha9895-X-a2n'
import IPython
import sys

# We will use IPython to fetch the notebook cell content dynamically and run it
session = IPython.get_ipython()
if session is not None:
    # Locate the cell content by searching the history or execution records
    # or execute the code directly
    try:
        cell_code = session.user_ns.get('_ih', [])[-1]
    except Exception:
        pass

# Defining the exact functions here so they are loaded successfully
print("Defining functions from cell Ha9895-X-a2n...")


Defining functions from cell Ha9895-X-a2n...


In [ ]:
import numpy as np
import pandas as pd
import warnings
import gc
import os

warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1','aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo','dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']

MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
 'DGS1', 'DGS5', 'DGS10']

CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative',
                    'volatility', 'ln_volume','div_payout','stock_split']

def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    split_labels = []
    for date in df['Date']:
        if date <= train_end:
            split_labels.append('train')
        elif date <= val_end:
            split_labels.append('validation')
        else:
            split_labels.append('test')
    df['split'] = split_labels
    return df

def clean_dataframe_from_inf_and_nan(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    return df

def fit_decision_tree(X, y, max_depth=20):
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf.fit(X, y)
    return clf, X.columns

def tpr_fpr_dataframe(y_true, y_pred, only_even=False):
    scores = []
    thresholds = np.linspace(0, 1, 51) if only_even else np.linspace(0, 1, 101)
    for t in thresholds:
        actual_positive = (y_true == 1)
        actual_negative = (y_true == 0)
        predict_positive = (y_pred >= t)
        predict_negative = (y_pred < t)
        tp = (predict_positive & actual_positive).sum()
        tn = (predict_negative & actual_negative).sum()
        fp = (predict_positive & actual_negative).sum()
        fn = (predict_negative & actual_positive).sum()
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1_score = 2*precision*recall / (precision+recall) if precision+recall > 0 else 0
        accuracy = (tp+tn) / (tp+tn+fp+fn)
        scores.append((t, tp, fp, fn, tn, precision, recall, accuracy, f1_score))
    columns = ['threshold', 'tp', 'fp', 'fn', 'tn','precision','recall', 'accuracy','f1_score']
    df_scores = pd.DataFrame(scores, columns=columns)
    df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
    df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)
    return df_scores

def load_and_prepare_data(data_path, use_ticker=True, use_month_wom=True, redefine_target_threshold=None):
    df_full = pd.read_parquet(data_path)
    GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
    CATEGORICAL = ['Month', 'Weekday', 'ticker_type']
    if use_ticker:
        CATEGORICAL.insert(2, 'Ticker')
    df_full['ln_volume'] = df_full.Volume.apply(lambda x: np.log(x+ 1e-6))
    df_full['div_payout'] = (df_full.Dividends>0).astype(int)
    df_full['stock_split'] = (df_full['Stock Splits']>0).astype(int)
    TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]
    NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO
    if redefine_target_threshold is not None:
        df_full['is_positive_growth_30d_future'] = (df_full['growth_future_30d'] > redefine_target_threshold).astype(int)
    df = df_full[df_full.Date>='2000-01-01'].copy()
    df.loc[:,'Month'] = pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
    df.loc[:,'Weekday'] = df['Weekday'].astype('string')
    if use_month_wom:
        df.loc[:,'wom'] = df.Date.apply(lambda d: (d.day-1)//7 + 1)
        df.loc[:,'wom'] = df.loc[:,'wom'].astype(str)
        df.loc[:,'month_wom'] = df.Month + '_w' + df.wom
        CATEGORICAL.append('month_wom')
    dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
    DUMMIES = dummy_variables.keys().to_list()
    df_with_dummies = pd.concat([df, dummy_variables], axis=1)
    min_date_df = df_with_dummies.Date.min()
    max_date_df = df_with_dummies.Date.max()
    df_with_dummies = temporal_split(df_with_dummies, min_date=min_date_df, max_date=max_date_df)
    return df_with_dummies, NUMERICAL, DUMMIES, NUMERICAL + DUMMIES

def add_manual_predictions(new_df):
    new_df['pred0_manual_cci'] = (new_df.cci>200).astype(int)
    new_df['pred1_manual_prev_g1'] = (new_df.growth_30d>1).astype(int)
    new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)
    new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <= 4.5) & (new_df['DGS5'] <= 4)).astype(int)
    new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS'] <= 4.795)).astype(int)
    return new_df

def train_models_and_predict(new_df, features_list, add_rf_thresholds=None):
    to_predict = 'is_positive_growth_30d_future'
    train_valid_df = new_df[new_df.split.isin(['train','validation'])]
    test_df = new_df[new_df.split.isin(['test'])]
    X_train_valid = clean_dataframe_from_inf_and_nan(train_valid_df[features_list+[to_predict]].copy(deep=True))
    X_test = clean_dataframe_from_inf_and_nan(test_df[features_list+[to_predict]].copy(deep=True))
    X_all = clean_dataframe_from_inf_and_nan(new_df[features_list+[to_predict]].copy(deep=True))
    y_train_valid = X_train_valid[to_predict]
    y_test = X_test[to_predict]
    X_train_valid_features = X_train_valid[features_list]
    X_test_features = X_test[features_list]
    X_all_features = X_all[features_list]
    clf_10, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=10)
    new_df['pred5_clf_10'] = clf_10.predict(X_all_features)
    clf_4, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=4)
    new_df['pred6_clf_4'] = clf_4.predict(X_all_features)
    clf_best, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=7)
    new_df['pred7_clf_second_best'] = clf_best.predict(X_all_features)
    rf_best = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1)
    rf_best = rf_best.fit(X_train_valid_features, y_train_valid)
    y_pred_all_clf = clf_best.predict_proba(X_all_features)
    y_pred_all_clf_class1 = np.array([k[1] for k in y_pred_all_clf])
    new_df['pred8_clf_second_best_rule_56'] = (y_pred_all_clf_class1 >= 0.56).astype(int)
    new_df['pred9_clf_second_best_rule_66'] = (y_pred_all_clf_class1 >= 0.66).astype(int)
    y_pred_all_rf = rf_best.predict_proba(X_all_features)
    y_pred_all_rf_class1 = np.array([k[1] for k in y_pred_all_rf])
    new_df['pred10_rf_best_rule_55'] = (y_pred_all_rf_class1 >= 0.55).astype(int)
    new_df['pred11_rf_best_rule_65'] = (y_pred_all_rf_class1 >= 0.65).astype(int)
    rf_tpr_df = None
    if add_rf_thresholds is not None:
        y_pred_test_rf = rf_best.predict_proba(X_test_features)
        y_pred_test_rf_class1 = np.array([k[1] for k in y_pred_test_rf])
        rf_tpr_df = tpr_fpr_dataframe(y_test, y_pred_test_rf_class1, only_even=True)
        pred_num = 12
        for threshold_int in add_rf_thresholds:
            new_df[f'pred{pred_num}_rf_best_rule_{threshold_int}'] = (y_pred_all_rf_class1 >= (threshold_int / 100.0)).astype(int)
            pred_num += 1
    return new_df, rf_tpr_df

def run_trading_simulations(new_df):
    PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
    sim1_results = []
    for pred in PREDICTIONS:
        pred_prefix = pred.split('_')[0]
        new_df['sim1_gross_rev_'+pred_prefix] = new_df[pred] * 100 * (new_df['growth_future_30d']-1)
        new_df['sim1_fees_'+pred_prefix] = -new_df[pred] * 100 * 0.002
        new_df['sim1_net_rev_'+pred_prefix] = new_df['sim1_gross_rev_'+pred_prefix] + new_df['sim1_fees_'+pred_prefix]
        filter_test_and_positive_pred = (new_df.split=='test')&(new_df[pred]==1)
        sim1_count_investments = len(new_df[filter_test_and_positive_pred])
        sim1_gross_rev = new_df[filter_test_and_positive_pred]['sim1_gross_rev_'+pred_prefix].sum()
        sim1_fees = new_df[filter_test_and_positive_pred]['sim1_fees_'+pred_prefix].sum()
        sim1_net_rev = new_df[filter_test_and_positive_pred]['sim1_net_rev_'+pred_prefix].sum()
        sim1_fees_percentage = -sim1_fees/sim1_gross_rev if sim1_gross_rev>0 else 0
        sim1_average_net_revenue = sim1_net_rev/sim1_count_investments if sim1_count_investments>0 else 0
        df_investments_count_daily = pd.DataFrame(new_df[filter_test_and_positive_pred].groupby('Date')[pred].count())
        sim1_avg_investments_per_day = df_investments_count_daily[pred].mean() if len(df_investments_count_daily) > 0 else 0
        sim1_q75_investments_per_day = df_investments_count_daily[pred].quantile(0.75) if len(df_investments_count_daily) > 0 else 0
        sim1_capital = 100 * 30 * sim1_q75_investments_per_day if sim1_q75_investments_per_day > 0 else 100
        sim1_CAGR = ((sim1_capital+sim1_net_rev)/sim1_capital)**(1/4) if sim1_capital > 0 else 0
        sim1_results.append((pred, sim1_count_investments, sim1_gross_rev, sim1_fees, sim1_net_rev,
                             sim1_fees_percentage, sim1_average_net_revenue, sim1_avg_investments_per_day,
                             sim1_capital, sim1_CAGR))
    columns_simulation = ['prediction', 'sim1_count_investments', 'sim1_gross_rev', 'sim1_fees', 'sim1_net_rev',
                           'sim1_fees_percentage','sim1_average_net_revenue','sim1_avg_investments_per_day',
                           'sim1_capital','sim1_CAGR']
    return pd.DataFrame(sim1_results, columns=columns_simulation)

def answer_q1(data_path):
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(data_path, use_ticker=False, use_month_wom=False)
    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list)
    df_sim1_results = run_trading_simulations(new_df)
    pred10_result = df_sim1_results[df_sim1_results.prediction == 'pred10_rf_best_rule_55']
    return pred10_result['sim1_CAGR'].values[0] if len(pred10_result) > 0 else None

def answer_q2(data_path):
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(data_path, use_ticker=False, use_month_wom=False)
    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list, add_rf_thresholds=list(range(56, 65)))
    df_sim1_results = run_trading_simulations(new_df)
    rf_results = df_sim1_results[df_sim1_results.prediction.str.contains('rf_best_rule')]
    rf_under5 = rf_results[rf_results.sim1_avg_investments_per_day < 5].sort_values('sim1_CAGR', ascending=False)
    if len(rf_under5) > 0:
        return int(rf_under5.iloc[0]['prediction'].split('rule_')[-1]) / 100.0
    return None

def answer_q3(data_path):
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(data_path, use_ticker=True, use_month_wom=True, redefine_target_threshold=1.022)
    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list)
    df_sim1_results = run_trading_simulations(new_df)
    pred5_result = df_sim1_results[df_sim1_results.prediction == 'pred5_clf_10']
    return int(pred5_result['sim1_count_investments'].values[0]) if len(pred5_result) > 0 else None

def answer_q4():
    return 'Confidence-Weighted Position Sizing & Multi-Model Consensus'

print('All functions and variables have been loaded successfully!')

All functions and variables have been loaded successfully!


Complete run below

In [1]:
!gdown --id 1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5 -O /content/stocks_df_combined_2026_09_18.parquet.brotli

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5
From (redirected): https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5&confirm=t&uuid=79f71e4f-cb82-4ae4-9b30-b7d37858ccdb
To: /content/stocks_df_combined_2026_09_18.parquet.brotli
100% 144M/144M [00:02<00:00, 55.2MB/s]


In [2]:
import os

file_path = 'stocks_df_combined_2026_09_18.parquet.brotli'
absolute_path = '/content/stocks_df_combined_2026_09_18.parquet.brotli'

print(f"File exists in current directory: {os.path.exists(file_path)}")
print(f"File exists at absolute path: {os.path.exists(absolute_path)}")

if os.path.exists(absolute_path):
    size_mb = os.path.getsize(absolute_path) / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")
else:
    print("Warning: File not found.")

File exists in current directory: True
File exists at absolute path: True
File size: 137.18 MB


In [3]:
import numpy as np
import pandas as pd
import warnings
import gc
import os

warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1','aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo','dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']

MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
 'DGS1', 'DGS5', 'DGS10']

CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative',
                    'volatility', 'ln_volume','div_payout','stock_split']

def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    split_labels = []
    for date in df['Date']:
        if date <= train_end:
            split_labels.append('train')
        elif date <= val_end:
            split_labels.append('validation')
        else:
            split_labels.append('test')
    df['split'] = split_labels
    return df

def clean_dataframe_from_inf_and_nan(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    return df

def fit_decision_tree(X, y, max_depth=20):
    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf.fit(X, y)
    return clf, X.columns

def tpr_fpr_dataframe(y_true, y_pred, only_even=False):
    scores = []
    thresholds = np.linspace(0, 1, 51) if only_even else np.linspace(0, 1, 101)
    for t in thresholds:
        actual_positive = (y_true == 1)
        actual_negative = (y_true == 0)
        predict_positive = (y_pred >= t)
        predict_negative = (y_pred < t)
        tp = (predict_positive & actual_positive).sum()
        tn = (predict_negative & actual_negative).sum()
        fp = (predict_positive & actual_negative).sum()
        fn = (predict_negative & actual_positive).sum()
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1_score = 2*precision*recall / (precision+recall) if precision+recall > 0 else 0
        accuracy = (tp+tn) / (tp+tn+fp+fn)
        scores.append((t, tp, fp, fn, tn, precision, recall, accuracy, f1_score))
    columns = ['threshold', 'tp', 'fp', 'fn', 'tn','precision','recall', 'accuracy','f1_score']
    df_scores = pd.DataFrame(scores, columns=columns)
    df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
    df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)
    return df_scores

def load_and_prepare_data(data_path, use_ticker=True, use_month_wom=True, redefine_target_threshold=None):
    df_full = pd.read_parquet(data_path)
    GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
    CATEGORICAL = ['Month', 'Weekday', 'ticker_type']
    if use_ticker:
        CATEGORICAL.insert(2, 'Ticker')
    df_full['ln_volume'] = df_full.Volume.apply(lambda x: np.log(x+ 1e-6))
    df_full['div_payout'] = (df_full.Dividends>0).astype(int)
    df_full['stock_split'] = (df_full['Stock Splits']>0).astype(int)
    TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]
    NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO
    if redefine_target_threshold is not None:
        df_full['is_positive_growth_30d_future'] = (df_full['growth_future_30d'] > redefine_target_threshold).astype(int)
    df = df_full[df_full.Date>='2000-01-01'].copy()
    df.loc[:,'Month'] = pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
    df.loc[:,'Weekday'] = df['Weekday'].astype('string')
    if use_month_wom:
        df.loc[:,'wom'] = df.Date.apply(lambda d: (d.day-1)//7 + 1)
        df.loc[:,'wom'] = df.loc[:,'wom'].astype(str)
        df.loc[:,'month_wom'] = df.Month + '_w' + df.wom
        CATEGORICAL.append('month_wom')
    dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
    DUMMIES = dummy_variables.keys().to_list()
    df_with_dummies = pd.concat([df, dummy_variables], axis=1)
    min_date_df = df_with_dummies.Date.min()
    max_date_df = df_with_dummies.Date.max()
    df_with_dummies = temporal_split(df_with_dummies, min_date=min_date_df, max_date=max_date_df)
    return df_with_dummies, NUMERICAL, DUMMIES, NUMERICAL + DUMMIES

def add_manual_predictions(new_df):
    new_df['pred0_manual_cci'] = (new_df.cci>200).astype(int)
    new_df['pred1_manual_prev_g1'] = (new_df.growth_30d>1).astype(int)
    new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)
    new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <= 4.5) & (new_df['DGS5'] <= 4)).astype(int)
    new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS'] <= 4.795)).astype(int)
    return new_df

def train_models_and_predict(new_df, features_list, add_rf_thresholds=None):
    to_predict = 'is_positive_growth_30d_future'
    train_valid_df = new_df[new_df.split.isin(['train','validation'])]
    test_df = new_df[new_df.split.isin(['test'])]
    X_train_valid = clean_dataframe_from_inf_and_nan(train_valid_df[features_list+[to_predict]].copy(deep=True))
    X_test = clean_dataframe_from_inf_and_nan(test_df[features_list+[to_predict]].copy(deep=True))
    X_all = clean_dataframe_from_inf_and_nan(new_df[features_list+[to_predict]].copy(deep=True))
    y_train_valid = X_train_valid[to_predict]
    y_test = X_test[to_predict]
    X_train_valid_features = X_train_valid[features_list]
    X_test_features = X_test[features_list]
    X_all_features = X_all[features_list]
    clf_10, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=10)
    new_df['pred5_clf_10'] = clf_10.predict(X_all_features)
    clf_4, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=4)
    new_df['pred6_clf_4'] = clf_4.predict(X_all_features)
    clf_best, _ = fit_decision_tree(X=X_train_valid_features, y=y_train_valid, max_depth=7)
    new_df['pred7_clf_second_best'] = clf_best.predict(X_all_features)
    rf_best = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1)
    rf_best = rf_best.fit(X_train_valid_features, y_train_valid)
    y_pred_all_clf = clf_best.predict_proba(X_all_features)
    y_pred_all_clf_class1 = np.array([k[1] for k in y_pred_all_clf])
    new_df['pred8_clf_second_best_rule_56'] = (y_pred_all_clf_class1 >= 0.56).astype(int)
    new_df['pred9_clf_second_best_rule_66'] = (y_pred_all_clf_class1 >= 0.66).astype(int)
    y_pred_all_rf = rf_best.predict_proba(X_all_features)
    y_pred_all_rf_class1 = np.array([k[1] for k in y_pred_all_rf])
    new_df['pred10_rf_best_rule_55'] = (y_pred_all_rf_class1 >= 0.55).astype(int)
    new_df['pred11_rf_best_rule_65'] = (y_pred_all_rf_class1 >= 0.65).astype(int)
    rf_tpr_df = None
    if add_rf_thresholds is not None:
        y_pred_test_rf = rf_best.predict_proba(X_test_features)
        y_pred_test_rf_class1 = np.array([k[1] for k in y_pred_test_rf])
        rf_tpr_df = tpr_fpr_dataframe(y_test, y_pred_test_rf_class1, only_even=True)
        pred_num = 12
        for threshold_int in add_rf_thresholds:
            new_df[f'pred{pred_num}_rf_best_rule_{threshold_int}'] = (y_pred_all_rf_class1 >= (threshold_int / 100.0)).astype(int)
            pred_num += 1
    return new_df, rf_tpr_df

def run_trading_simulations(new_df):
    PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
    sim1_results = []
    for pred in PREDICTIONS:
        pred_prefix = pred.split('_')[0]
        new_df['sim1_gross_rev_'+pred_prefix] = new_df[pred] * 100 * (new_df['growth_future_30d']-1)
        new_df['sim1_fees_'+pred_prefix] = -new_df[pred] * 100 * 0.002
        new_df['sim1_net_rev_'+pred_prefix] = new_df['sim1_gross_rev_'+pred_prefix] + new_df['sim1_fees_'+pred_prefix]
        filter_test_and_positive_pred = (new_df.split=='test')&(new_df[pred]==1)
        sim1_count_investments = len(new_df[filter_test_and_positive_pred])
        sim1_gross_rev = new_df[filter_test_and_positive_pred]['sim1_gross_rev_'+pred_prefix].sum()
        sim1_fees = new_df[filter_test_and_positive_pred]['sim1_fees_'+pred_prefix].sum()
        sim1_net_rev = new_df[filter_test_and_positive_pred]['sim1_net_rev_'+pred_prefix].sum()
        sim1_fees_percentage = -sim1_fees/sim1_gross_rev if sim1_gross_rev>0 else 0
        sim1_average_net_revenue = sim1_net_rev/sim1_count_investments if sim1_count_investments>0 else 0
        df_investments_count_daily = pd.DataFrame(new_df[filter_test_and_positive_pred].groupby('Date')[pred].count())
        sim1_avg_investments_per_day = df_investments_count_daily[pred].mean() if len(df_investments_count_daily) > 0 else 0
        sim1_q75_investments_per_day = df_investments_count_daily[pred].quantile(0.75) if len(df_investments_count_daily) > 0 else 0
        sim1_capital = 100 * 30 * sim1_q75_investments_per_day if sim1_q75_investments_per_day > 0 else 100
        sim1_CAGR = ((sim1_capital+sim1_net_rev)/sim1_capital)**(1/4) if sim1_capital > 0 else 0
        sim1_results.append((pred, sim1_count_investments, sim1_gross_rev, sim1_fees, sim1_net_rev,
                             sim1_fees_percentage, sim1_average_net_revenue, sim1_avg_investments_per_day,
                             sim1_capital, sim1_CAGR))
    columns_simulation = ['prediction', 'sim1_count_investments', 'sim1_gross_rev', 'sim1_fees', 'sim1_net_rev',
                           'sim1_fees_percentage','sim1_average_net_revenue','sim1_avg_investments_per_day',
                           'sim1_capital','sim1_CAGR']
    return pd.DataFrame(sim1_results, columns=columns_simulation)

def answer_q1(data_path):
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(data_path, use_ticker=False, use_month_wom=False)
    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list)
    df_sim1_results = run_trading_simulations(new_df)
    pred10_result = df_sim1_results[df_sim1_results.prediction == 'pred10_rf_best_rule_55']
    return pred10_result['sim1_CAGR'].values[0] if len(pred10_result) > 0 else None

def answer_q2(data_path):
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(data_path, use_ticker=False, use_month_wom=False)
    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list, add_rf_thresholds=list(range(56, 65)))
    df_sim1_results = run_trading_simulations(new_df)
    rf_results = df_sim1_results[df_sim1_results.prediction.str.contains('rf_best_rule')]
    rf_under5 = rf_results[rf_results.sim1_avg_investments_per_day < 5].sort_values('sim1_CAGR', ascending=False)
    if len(rf_under5) > 0:
        return int(rf_under5.iloc[0]['prediction'].split('rule_')[-1]) / 100.0
    return None

def answer_q3(data_path):
    new_df, NUMERICAL, DUMMIES, features_list = load_and_prepare_data(data_path, use_ticker=True, use_month_wom=True, redefine_target_threshold=1.022)
    new_df = add_manual_predictions(new_df)
    new_df, _ = train_models_and_predict(new_df, features_list)
    df_sim1_results = run_trading_simulations(new_df)
    pred5_result = df_sim1_results[df_sim1_results.prediction == 'pred5_clf_10']
    return int(pred5_result['sim1_count_investments'].values[0]) if len(pred5_result) > 0 else None

def answer_q4():
    return 'Confidence-Weighted Position Sizing & Multi-Model Consensus'

print('All functions and variables have been loaded successfully!')

All functions and variables have been loaded successfully!


In [4]:
# Re-running the homework script using functions from the main cell.
# Make sure you have run the cell above (Ha9895-X-a2n) first to define the functions!
data_path = '/content/stocks_df_combined_2026_09_18.parquet.brotli'

try:
    answers = {}
    answers['Q1'] = answer_q1(data_path)
    answers['Q2'] = answer_q2(data_path)
    answers['Q3'] = answer_q3(data_path)
    answers['Q4'] = answer_q4()

    print("\n" + "="*70)
    print("FINAL ANSWERS SUMMARY")
    print("="*70)
    if 'Q1' in answers and answers['Q1'] is not None:
        print(f"  Q1 (CAGR for pred10_rf_best_rule_55 with fewer features): {answers['Q1']:.4f}")
    if 'Q2' in answers and answers['Q2'] is not None:
        print(f"  Q2 (RF threshold with avg trades/day < 5):               {answers['Q2']:.2f}")
    if 'Q3' in answers and answers['Q3'] is not None:
        print(f"  Q3 (Number of trades for pred5_clf_10 with new target):   {answers['Q3']}")
except NameError:
    print("NameError: The functions are not defined. Please run the cell above (Ha9895-X-a2n) first, and then run this cell.")


FINAL ANSWERS SUMMARY
  Q1 (CAGR for pred10_rf_best_rule_55 with fewer features): 1.1359
  Q2 (RF threshold with avg trades/day < 5):               0.56
  Q3 (Number of trades for pred5_clf_10 with new target):   7337
